## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from umap import UMAP

from analysis_helpers import Episode, Participant

## Classes/functions

In [2]:
class Point:
    def __init__(self, coord=None):
        self.coord = np.array(coord)

In [3]:
class LineSegment:  
    def __init__(self, p1=None, p2=None):
        if not isinstance(p1, Point):
            p1 = Point(p1)
        if not isinstance(p2, Point):
            p2 = Point(p2)
        
        self.p1 = p1
        self.p2 = p2
        self.vec = self.p2.coord - self.p1.coord
        self.norm = self.vec / np.linalg.norm(self.vec)
    
    @property
    def angle(self):
        p1 = np.zeros_like(self.get_p1())
        p2 = np.zeros_like(self.get_p1())
        p2[0] = 1
        ref = LineSegment(p1, p2)
        return self.angle_with(ref)
    
    def get_p1(self):
        return self.p1.coord
    
    def get_p2(self):
        return self.p2.coord
        
    def intersects(self, z):
        Q = circ.get_center()
        r = circ.get_radius()
        P1 = self.get_p1()
        V = self.get_p2() - P1

        a = V.dot(V)
        b = 2 * V.dot(P1 - Q)
        c = P1.dot(P1) + Q.dot(Q) - 2 * P1.dot(Q) - r ** 2

        disc = b ** 2 - 4 * a * c
        if disc < 0:
            return False

        sqrt_disc = math.sqrt(disc)
        t1 = (-b + sqrt_disc) / (2 * a)
        t2 = (-b - sqrt_disc) / (2 * a)
        if not (0 <= t1 <= 1 or 0 <= t2 <= 1):
            return False
    
        return True
        
    def angle_with(self, ref):
        assert isinstance(ref, LineSegment)
        v0 = ref.vec
        v1 = self.vec
        angle = np.arccos(v0.dot(v1) / (np.linalg.norm(v0) * np.linalg.norm(v1)))
        if self.vec[1] < 0:
            angle = (2 * np.pi) - angle
            
        return angle

In [4]:
class Circle:
    def __init__(self, center=None, r=None):
        self.center = np.array(center)
        self.r = r 
    
    def get_center(self):
        return self.center
    
    def get_radius(self):
        return self.r

In [5]:
def compute_coord(xi, yi, w, seglist):
    z = Circle(center=[xi,yi], r=w)
        
    segs = list(filter(lambda s: s.intersects(z), seglist))
    c = len(segs)
    if c > 1:
        u, v  = np.array([seg.norm for seg in segs]).mean(0)
        rads = np.array([seg.angle for seg in segs])
        p, z = rayleigh(rads)
    else:
        u = 0
        v = 0
        p = 1
    return u, v, p, c

In [6]:
def add_arrows(axes, x, y, **kwargs):
    # spacing of arrows
    aspace = .05 * GRID_SCALE
    # distance spanned between pairs of points
    r = [0]
    for i in range(1, len(x)):
        dx = x[i] - x[i - 1]
        dy = y[i] - y[i - 1]
        r.append(np.sqrt(dx * dx + dy * dy))

    r = np.array(r)
    # cumulative sum of r, used to save time
    rtot = []
    for i in range(len(r)):
        rtot.append(r[0:i].sum())
    rtot.append(r.sum())
    # will hold tuple(x, y, theta) for each arrow
    arrow_data = []
    # current point on walk along data
    arrow_pos = 0
    rcount = 1
    while arrow_pos < r.sum():
        x1, x2 = x[rcount - 1], x[rcount]
        y1, y2 = y[rcount - 1], y[rcount]
        da = arrow_pos - rtot[rcount]
        theta = np.arctan2((x2 - x1), (y2 - y1))
        ax = np.sin(theta) * da + x1
        ay = np.cos(theta) * da + y1
        arrow_data.append((ax, ay, theta))
        arrow_pos += aspace
        while arrow_pos > rtot[rcount + 1]:
            rcount += 1
            if arrow_pos > rtot[-1]:
                break

    for ax, ay, theta in arrow_data:
        # use aspace as a guide for size and length of things
        # scaling factors were chosen by experimenting a bit
        axes.arrow(ax,
                   ay,
                   np.sin(theta) * aspace / 10,
                   np.cos(theta) * aspace / 10,
                   head_width=aspace / 3,
                   **kwargs)

## Load data

In [7]:
atlep1 = Episode('atlep1')
participants = Participant.load_all()
avg_participant = Participant('average')

## Project events to 2D

In [19]:
UMAP_PARAMS = {
    'n_components': 2,
    'metric': 'cosine',
    'output_metric': 'euclidean',
    'random_state': 0,
    'verbose': True
}

In [20]:
to_reduce = [atlep1.events]
for rectype in ('atlep1', 'delayed'):
    to_reduce.append(avg_participant.events[rectype])
    for p in participants:
        to_reduce.append(p.events[rectype])
        
split_inds = np.cumsum([vec.shape[0] for vec in to_reduce])[:-1]

reducer = UMAP(**UMAP_PARAMS)
embeddings = reducer.fit_transform(np.vstack(to_reduce))

split_embeddings = np.vsplit(embeddings, split_inds)


atlep1.path_2d = split_embeddings[0]
split_ix = 1
for rectype in ('atlep1', 'delayed'):
    avg_participant.paths_2d[rectype] = split_embeddings[split_ix]
    split_ix += 1
    for p in participants:
        p.paths_2d[rectype] = split_embeddings[split_ix]
        split_ix += 1

/opt/conda/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=0, verbose=True)
Mon Jun 22 21:39:35 2026 Construct fuzzy simplicial set
Mon Jun 22 21:39:39 2026 Finding Nearest Neighbors
Mon Jun 22 21:39:39 2026 Finished Nearest Neighbor Search
Mon Jun 22 21:39:39 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Mon Jun 22 21:39:41 2026 Finished embedding


In [44]:
embeddings.min(axis=0), embeddings.max(axis=0)

(array([-5.3969274, -1.5303159], dtype=float32),
 array([15.215732, 14.198397], dtype=float32))

In [ ]:
# create 2D grid
scale = np.abs(episode).max()
step = scale / 25
X, Y = np.meshgrid(np.arange(-scale, scale, step), np.arange(-scale, scale, step))

# turn embedded recall event model into a list of line segments
seglist = []
for i, (turkid, sub_emb) in enumerate(recalls.items()):
    for j in range(sub_emb.shape[0] - 1):
        p1 = Point(coord=sub_emb[j, :])
        p2 = Point(coord=sub_emb[j + 1, :])
        seg = LineSegment(p1=p1, p2=p2)

        seglist.append(seg)

# compute the average vector and p-value at each grid point
U = np.zeros_like(X)
V = np.zeros_like(X)
P = np.zeros_like(X)
# Z = np.zeros_like(X)
C = np.zeros_like(X)
for i, (x, y) in enumerate(zip(X, Y)):
    for j, (xi, yi) in enumerate(zip(x, y)):
        U[i, j], V[i, j], P[i, j], C[i, j] = compute_coord(xi, yi, step * 2, seglist, kind='circle')

# multiple comparisons correction
thresh = .001
Pc = mt(P.ravel(), method='fdr_bh', alpha=.05)[1].reshape(np.shape(X))
M = np.hypot(U, V)
M = plt.cm.Blues(M)
M[Pc > thresh] = [.5, .5, .5, .1]
M[Pc == 1] = [.5, .5, .5, 0]

# create figure with subplots
plt.figure(figsize=(12, (len(recalls) // 8) * 2))
mpl.rcParams['pdf.fonttype'] = 42
axarr = [0 for i in range(2)]

axarr[0] = plt.subplot2grid((len(recalls) // 8 + 3, 8), (0, 1), colspan=3, rowspan=2)
axarr[1] = plt.subplot2grid((len(recalls) // 8 + 3, 8), (0, 4), colspan=3, rowspan=2)

for i in range(2, (len(recalls) // 8 + 3)):
    for j in range(0, 8):
        ax = plt.subplot2grid((len(recalls) // 8 + 3, 8), (i, j))
        axarr.append(ax)

# plot episode trajectory and events
axarr[0].scatter(episode[:, 0], episode[:, 1], c=range(episode.shape[0]),
                 cmap=cmap, s=150, zorder=3)
axarr[0].scatter(episode[:, 0], episode[:, 1], c='k', cmap=cmap, s=200, zorder=2)
axarr[0].plot(episode[:, 0], episode[:, 1], zorder=1, c='k', alpha=.5)
add_arrows(axarr[0], episode[:, 0], episode[:, 1], zorder=0, alpha=1, color='k', fill=True)
axarr[0].set_title('Episode events')
axarr[0].set_xlim(episode.min(0)[0] - 1, episode.max(0)[0] + 1)
axarr[0].set_ylim(episode.min(0)[1] - 1, episode.max(0)[1] + 1)
axarr[0].text(0, 1, 'A', horizontalalignment='center', transform=axarr[0].transAxes, fontsize=18)

# plot average recall events
axarr[1].quiver(X, Y, U, V, color=M.reshape(M.shape[0] * M.shape[1], 4), zorder=1, width=.004)
axarr[1].plot(avg_recall[:, 0], avg_recall[:, 1], zorder=2, c='k', alpha=1)
axarr[1].plot(episode[:, 0], episode[:, 1], zorder=1, c='k', alpha=.5)
add_arrows(axarr[1], avg_recall[:, 0], avg_recall[:, 1], zorder=3, alpha=1, color='k', fill=True)
axarr[1].scatter(avg_recall[:, 0], avg_recall[:, 1], c=range(avg_recall.shape[0]), cmap=cmap,
                 s=150, zorder=4)
axarr[1].scatter(avg_recall[:, 0], avg_recall[:, 1], c='k', cmap=cmap, s=200, zorder=3)
axarr[1].set_title('Average recall events')
axarr[1].set_xlim(episode.min(0)[0] - 1, episode.max(0)[0] + 1)
axarr[1].set_ylim(episode.min(0)[1] - 1, episode.max(0)[1] + 1)
axarr[1].text(0, 1, 'B',
              horizontalalignment='center',
              transform=axarr[1].transAxes,
              fontsize=18)

# plot individual recalls
axarr[2].text(0, 1.05, 'C', horizontalalignment='center', transform=axarr[2].transAxes, fontsize=18)

if rectype == 'atlep1':
    ids = id_maps['session 1']
elif rectype == 'delayed':
    ids = id_maps['session 2']
elif rectype == 'atlep2':
    ids = id_maps.loc[id_maps.index.str.contains('A'), 'session 2']
else:
    ids = id_maps.loc[id_maps.index.str.contains('B'), 'session 2']

for i, turkid in enumerate(ids, start=2):
    rec_emb = recalls[turkid]
    m = mappings[np.where(mappings.T[0] == turkid)].ravel()[1]
    axarr[i].scatter(rec_emb[:, 0], rec_emb[:, 1], c=cmap(m / len(episode)), cmap=cmap, s=60, zorder=2)
    axarr[i].plot(rec_emb[:, 0], rec_emb[:, 1], zorder=1, c='k', alpha=.25)
    axarr[i].plot(avg_recall[:, 0], avg_recall[:, 1], zorder=3, c='k', alpha=1)
    add_arrows(axarr[i], rec_emb[:, 0], rec_emb[:, 1], zorder=1, alpha=.25, color='k', minifig=True, fill=True)
    axarr[i].set_xlim(episode.min(0)[0] - 1, episode.max(0)[0] + 1)
    axarr[i].set_ylim(episode.min(0)[1] - 1, episode.max(0)[1] + 1)
    axarr[i].set_title(f'P{i}')

for a in axarr:
    a.axis('off')

plt.tight_layout()
plt.subplots_adjust(wspace=0, hspace=0.25)
plt.savefig(opj(fig_dir, rectype, f'{np_seed}.pdf'))






In [ ]:
plt.figure(figsize=(10, 9))
axarr = [0, 0]
axarr[0] = plt.subplot2grid((5, 6), (0, 0), colspan=3, rowspan=2)
axarr[1] = plt.subplot2grid((5, 6), (0, 3), colspan=3, rowspan=2)
for i in range(2, 5):
    for j in range(0, 6):
        axarr.append(plt.subplot2grid((5, 6), (i, j)))

# ===== PANEL A =====
axarr[0].scatter(video_embedding[:, 0], 
                 video_embedding[:, 1], 
                 c=range(video_embedding.shape[0]), 
                 cmap=cmap, 
                 s=150, 
                 zorder=3)
axarr[0].scatter(video_embedding[:, 0], 
                 video_embedding[:, 1], 
                 c='k', 
                 cmap=cmap, 
                 s=200, 
                 zorder=2)
axarr[0].plot(video_embedding[:, 0], 
              video_embedding[:, 1], 
              zorder=1, 
              c='k', 
              alpha=.5)
add_arrows(axarr[0], 
           video_embedding[:, 0], 
           video_embedding[:, 1], 
           zorder=0, 
           alpha=.5, 
           color='k', 
           fill=True)
axarr[0].axis('off')
axarr[0].set_title('Video events')
axarr[0].set_xlim(-20, 21)
axarr[0].set_ylim(-17, 23)
axarr[0].text(0, 1, 'A',
              horizontalalignment='center',
              transform=axarr[0].transAxes,
              fontsize=18)

# ===== PANEL B =====
axarr[1].quiver(X, Y, U, V, 
                color=M.reshape(M.shape[0] * M.shape[1], 4), 
                zorder=1, 
                width=.004)
axarr[1].plot(avg_recall[:, 0], avg_recall[:, 1], zorder=2, c='k', alpha=.5)
add_arrows(axarr[1], 
           avg_recall[:, 0], 
           avg_recall[:, 1], 
           zorder=3, 
           alpha=1, 
           color='k', 
           fill=True)
axarr[1].scatter(avg_recall[:, 0], 
                 avg_recall[:, 1], 
                 c=range(avg_recall.shape[0]), 
                 cmap=cmap, 
                 s=150, 
                 zorder=4)
axarr[1].scatter(avg_recall[:, 0], 
                 avg_recall[:, 1], 
                 c='k', 
                 cmap=cmap, 
                 s=200, 
                 zorder=3)
axarr[1].axis('off')
axarr[1].set_title('Recalled events')
axarr[1].set_xlim(-20, 21)
axarr[1].set_ylim(-17, 23)
axarr[1].text(0, 1, 'B',
              horizontalalignment='center',
              transform=axarr[1].transAxes,
              fontsize=18)

# ===== PANEL C =====
for i, (e, m) in enumerate(zip(recall_embeddings, mappings)):
    ax = axarr[i + 2]
    ax.scatter(e[:, 0],
               e[:, 1], 
               c=cmap(m / video_embedding.shape[0]), 
               cmap=cmap, 
               s=100, 
               zorder=2)
    ax.plot(e[:, 0], e[:, 1], zorder=1, c='k', alpha=.25)
    add_arrows(ax, e[:, 0], e[:, 1], zorder=1, alpha=.25, color='k', fill=True)
    ax.plot(video_embedding[:, 0], video_embedding[:, 1], c='k', zorder=3)
    ax.axis('off')
    ax.set_xlim(-21, 23)
    ax.set_ylim(-18, 24)
    ax.set_title(f'P{i + 1}')

axarr[-1].axis('off')
axarr[2].text(0, 1.05, 'C',
              horizontalalignment='center',
              transform=axarr[2].transAxes,
              fontsize=18)

plt.tight_layout()
plt.subplots_adjust(wspace=.05, hspace=.25)
# plt.savefig(FIG_DIR.joinpath('trajectory.pdf'))
plt.show()